In [ ]:
# ===== REFINED TOP-3 CoV on ms640 @ budget 100,000 — CONFIG ==============
# ===== edit ONLY this cell ===============================================
#
# WHAT THIS RUNS. The same two-stage pipeline as cov_top3_ms640_abel.ipynb /
# _len.ipynb, with a refined ranking. For each of the 640 ms640 presentations
# Stage A enumerates every valid change-of-variables candidate and picks 3;
# Stage B searches all three in rank order at BUDGET nodes each. What changed is
# how the 3 are picked:
#
#   1. rank by the arm's key            (abel | abel+total length | total length)
#   2. then by S, the smaller mean block <- the tie-break, "_s" rules
#   3. drop any candidate that is an earlier pick with x and y renamed
#   4. pull deeper into the same ranking to refill the freed slot
#
# Step 3 is the point. The 8 signed permutations of {x, y} are automorphisms, so
# a solution exists for one iff it exists for all eight and the minimal AC path
# length is identical — a slot holding a relabel of an earlier pick re-searches
# a start this arm has already searched. The shipped abel manifest spends 723
# such slots over 500 of the 640 presentations; the shipped len manifest 424
# over 343. Every rule below spends none.
#
# WHAT THIS IS EXPECTED TO SHOW, so a null result is not read as a bug. On the
# frozen subset-60 sweeps (where every candidate of every row was searched, so a
# promoted pick can be priced without new search) the dedup is *paired-identical*
# at budget 1,000 and 10,000 on all three keys — 0 wins, 0 losses. It is hygiene:
# k should mean k distinct searches. Its one measured gain there is (total) at
# 10,000, top-3 49 -> 50. MK is the genuinely open question: it helps (abel) at
# 1,000 and HURTS it at 10,000, and S hurts that arm at both — but (abel) is not
# an arm anyone runs. On (abel, total) S and MK are indistinguishable. Subset-60 is 60 rows and those margins are 1-2
# rows wide, which the repo's own control-with-no-dynamic-range and gap-metric
# lessons say is not yet a property of the key. See COV_RELABEL_B1K.md.
#
# WHICH ARMS ARE WORTH THE NODES — read before queueing a session.
#
#   abel_rd        DO NOT RUN. Provably a no-op, settled today at zero search.
#                  The dedup keeps the first-ranked member of each relabel class,
#                  so it never changes rank 1 (verified: rank 1 identical on
#                  640/640 vs shipped abel), and shipped abel's rank 1 ALREADY
#                  SOLVES 640/640 at 100,000. Its solve count and its 458,688-node
#                  rank-1 bill are identical to the shipped arm by construction.
#   len_rd         RUN FIRST. The only arm with real dynamic range: len's rank 1
#                  fails on 7 presentations (425, 435, 573, 599, 601, 634, 635)
#                  and the dedup rewrites the top 3 on 5 of them.
#   abel_len_rd    the control for the third term. Run paired with the next one.
#   abel_len_rd_s  THE RULE — abel -> length -> S. The pair is powered on a cost
#                  comparison, NOT on solves, which are pinned at the 640/640
#                  ceiling for every abel-first arm at this budget.
#   len_rd_s       S on the length arm. Subset-60 likes this one: vs len_rd it is
#                  -171 nodes at budget 1,000 and -6,395 with top-3 +2 at 10,000,
#                  where MK manages +1,000 and -2,125 / +1.
#   *_rd_mk        MK instead of S, kept only as a comparison. On (abel, total) the
#                  two are indistinguishable (8 nodes apart on ~294k); on the
#                  length arm S is clearly better. Not the recommendation.
#
# So this run is a powered test of COST and of 17 rank-1 changes. It cannot be a
# powered test of the solve rate on an abel arm, because that metric has no
# headroom left on ms640.
#
# ALL THREE RANKS ALWAYS RUN, including ranks below one that already solved, so
# every rank is measured on the same 640 presentations; under early exit ranks
# 2-3 would exist only where rank 1 failed and their means would describe a
# harder, self-selected subset. The deployed early-exit cost is not lost —
# summarize() recovers it exactly as first_solve_nodes. A solve does NOT mark a
# presentation done, or a restart would skip precisely the easy ones.
#
# ARMS RUN SEQUENTIALLY IN ONE SESSION. RULES is a list and the RUN cell walks
# it in order, finishing one arm before starting the next. Each writes its own
# jsonl (the rule is in the filename) and none touches another's, so a restart
# resumes the arm that was in flight and skips the finished ones. To go faster,
# open a second Colab session on this notebook with the list split between them.
#
# RESTART CONTRACT. Runtime -> Restart, then Run All, continues this run: SETUP
# resets the repo to the latest push, purges the stale experiments.* modules,
# and seeds the local jsonl back from Drive; RUN resumes from it. Mid-run
# hotfixes must be pushed as .py files — a pushed .ipynb does NOT reach an
# already-open Colab notebook.

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "research/w5/stable-ac-escape"   # must match the actual git branch
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push

MOUNT_DRIVE = True           # mirror the results jsonl to Drive every few
                             # minutes + at the end, and seed it BACK on a fresh
                             # VM so resume continues where the last session
                             # stopped. The runner always writes locally
                             # (appending onto the Drive FUSE mount is unsafe).
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx_results/cov_top3"

# --- experiment knobs ------------------------------------------------------
# RULES is the only knob that differs between arms; the RUN cell walks the list
# in order, one arm fully finished before the next starts. Each arm writes its
# own jsonl (the rule is in the filename) and none touches another's, so a
# restart resumes whichever arm was in flight and skips the finished ones.
#
#   "abel_len_rd_s"   (abel, total, S)    THE RULE — abel -> length -> S
#   "len_rd_s"        (total, S)          the same third term on the length arm
#   "abel_len_rd"     (abel, total)       the control for the third term
#   "len_rd"          (total)             the SHIPPED len arm + dedup
#   "abel_rd"         (abel)              DO NOT RUN — provably a no-op
#   "*_rd_mk"         MK instead of S     comparison only, never the recommendation
#
# WHAT THE BUDGET-1,000 RUN OVER ALL 640 ALREADY SETTLED (2026-08-11), so this
# session is read against a prediction rather than fished for a story:
#
#   plain greedy (baseline truncated to 1,000)   top-3 554   122,208 nodes
#   abel                          rank 1 584   top-3 596   169k deployed / 249k as run
#   abel -> length -> S           rank 1 590   top-3 596   162k deployed / 334k as run
#   len                           rank 1 573   top-3 587   215k deployed / 315k as run
#   length -> S                   rank 1 572   top-3 588   213k deployed / 331k as run
#
# S buys rank 1 (+6 solves, -7,277 deployed nodes) and costs ranks 2-3, whose
# solve counts fall 580/580 -> 524/543: it promotes near-copies of the rank-1
# pick where the name tie-break promoted unlike starts. Both accountings are
# reported here for that reason. At 100,000 the abel arms have NO solve headroom
# (shipped abel's rank 1 already takes 640/640), so the abel pair is a test of
# COST and of the ~17 rank-1 changes; the length pair is where solves can move.
#
# ETA, from the two frozen 100,000-node runs on this same set: abel 1.16 CPU-h /
# 2.54M nodes, len 1.23 CPU-h / 3.14M nodes. The S arms search the same 640 x 3
# slots at the same budget with a different pick in slots 2-3, and at budget
# 1,000 that cost them ~1.3x more nodes, so budget ~1.5-2 CPU-h each — well
# under an hour of wall clock apiece once HIGH_SPEEDUP and the pool are on.
RULES       = ["abel_len_rd_s", "len_rd_s"]
BUDGET      = 100_000        # PER SEARCH; a presentation costs <= K * this
K           = 3              # ranks per presentation (the manifests are built at 3)
HIGH_SPEEDUP = True          # compact fast solver (~2.9x); result-neutral —
                             # a solved fast search is re-solved by the normal
                             # solver for its path, so every written row is
                             # identical to a slow-mode row and the files resume
                             # across the two modes
CHUNKS      = 1              # this arm is one session; raise it (with
CHUNK_INDEX = None           # CHUNK_INDEX = 1..CHUNKS) only to split ONE arm
                             # across more machines, then run the MERGE cell


In [ ]:
# ==================== SETUP (clone / pull / Drive) ========================
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy pyyaml")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)                      # relative paths + "import experiments…"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# a `git reset --hard` rewrites .py files but sys.modules keeps the OLD module
# objects -- drop them so RUN imports what SETUP just fetched (pull != reload)
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# --- Drive: mount + seed-back (fresh VM -> local resume state) -------------
import glob, shutil
LOCAL_OUT = os.path.join(REPO_ROOT, "results", "stable_ac", "cov", "cov_top3")
os.makedirs(LOCAL_OUT, exist_ok=True)
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for src in glob.glob(os.path.join(DRIVE_DIR, "*.jsonl")):
        dst = os.path.join(LOCAL_OUT, os.path.basename(src))
        # the bigger file wins: local mid-run state beats a stale mirror, and on
        # a fresh VM the mirror beats the (absent/empty) local file
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            shutil.copyfile(src, dst)
            print("seeded from Drive:", os.path.basename(src))


In [ ]:
# ==================== RUN =================================================
# Production budgets run HERE, never on the dev machine: the repo caps any
# locally-launched search at 1,000 nodes, and the runner enforces that cap
# unless this flag is set.
os.environ["ACSOLVERX_ALLOW_BIG"] = "1"

from experiments.stable_ac.cov.run import cov_top3_relabel as rd
from experiments.stable_ac.cov.run import cov_top3_run as R
from experiments.stable_ac.cov.run import cov_top3_relabel_run as rdrun

# THE GATES, for EVERY arm in RULES, before a single node is spent anywhere —
# an arm that would die on a bad manifest must die now and not two hours into
# the arm before it. Stage A is committed (1,920 picks over 640 presentations
# per rule); rebuild only if a shallow clone somehow lacks it, which explores
# ZERO nodes and takes ~3 s. The manifest path is DERIVED from the rule, never
# passed beside it: a stale path plus another rule's name is a wrong experiment
# wearing a plausible filename. preflight then re-derives every pick's relabel
# class from its own (r1, r2) rather than trusting the stored tag, because a
# deduped manifest path is still writable by an undeduped build and the rule
# name alone cannot detect that.
for _rule in RULES:
    _man = rd.manifest_path(_rule)
    if not os.path.exists(os.path.join(REPO_ROOT, _man)):
        print(f"manifest missing for {_rule} — rebuilding (no search)")
        rd.build(rule=_rule, out_path=_man)
    _pf = rdrun.preflight(_rule)
    print(f"[preflight] {_rule}: no relabel repeats over {_pf['n_pres']} presentations")

# mirror local jsonls -> Drive every 3 min (and once at the end). Whole-file
# copies of an append-only jsonl: a torn tail line is repaired on resume. The
# thread never prints (a background thread must not).
import threading
def _sync_to_drive():
    if not (IN_COLAB and MOUNT_DRIVE):
        return
    for src in glob.glob(os.path.join(LOCAL_OUT, "*.jsonl")):
        dst = os.path.join(DRIVE_DIR, os.path.basename(src))
        # size-monotonic: an append-only jsonl only ever grows, so never
        # overwrite a bigger Drive copy with a smaller local one (a session
        # holding a stale seeded copy of another arm must not clobber it)
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            tmp = dst + ".tmp"
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
def _mirror_loop():
    while not _mirror_stop.wait(180):
        try: _sync_to_drive()
        except Exception: pass                 # transient Drive hiccup: next tick
_mirror_stop = threading.Event()
threading.Thread(target=_mirror_loop, daemon=True).start()

# One arm at a time, in the order RULES lists them, each finished before the
# next starts. A crash or a Colab disconnect loses no finished arm: every arm
# has its own jsonl and R.run resumes from it, so Restart -> Run All picks up
# mid-arm and skips the completed ones.
#
# `registered` splices the rule into the shipped whitelist for the duration of
# that arm and takes it back out afterwards, including on error. It is NOT
# permanent: cov_top3_manifest.RULES is read at module scope elsewhere (tests
# parametrize on it at collection time, and its no-argument CLI builds its
# build-everything list from it, which would overwrite these manifests with
# undeduped ones). run / summarize / merge_chunks all validate through
# load_config, so all of them belong inside the block.
try:
    for RULE in RULES:
        MANIFEST = rd.manifest_path(RULE)
        groups = rd.load_manifest(MANIFEST, rule=RULE)
        print(f"\n===== {RULE}: {len(groups)} presentations x <= {K} ranks "
              f"@ budget {BUDGET:,} =====")
        with rd.registered(RULE):
            out_path = R.run(rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                             chunk_index=CHUNK_INDEX, high_speedup=HIGH_SPEEDUP,
                             manifest=MANIFEST)
            # This arm's score. The plain-greedy controls are read by TRUNCATING
            # the frozen 1,000,000-node ms640 baseline (zero new search) at
            # BUDGET and at K x BUDGET, and the paired nodes/path comparison runs
            # over the presentations BOTH arms solved. Then the gate: every
            # search that overlaps the frozen 10,000-node subset-60 sweep must
            # reproduce it node for node.
            R.summarize(out_path, rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                        chunk_index=CHUNK_INDEX, manifest=MANIFEST)
        _sync_to_drive()                       # this arm is done — mirror it now
        print(f"[{RULE}] written to {out_path}")
finally:
    _mirror_stop.set()
    _sync_to_drive()                           # final sync, incl. the last rows
    if IN_COLAB and MOUNT_DRIVE:
        print("mirrored to", DRIVE_DIR)


In [ ]:
# ============ MERGE / COMPARE (optional, after the other arms finish) ======
# Separate cell because each half needs a file this session did not write: run
# it only once the run(s) it names have printed their "done" line and mirrored
# to Drive. It re-seeds from Drive first, so run it in whichever session you like.
for src in glob.glob(os.path.join(DRIVE_DIR, "*.jsonl")) if (IN_COLAB and MOUNT_DRIVE) else []:
    dst = os.path.join(LOCAL_OUT, os.path.basename(src))
    if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
        shutil.copyfile(src, dst)

# (a) only if you split THIS arm across several machines (CHUNKS > 1)
if CHUNKS > 1:
    with rd.registered(RULE):
        out_path = R.merge_chunks(rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                                  manifest=MANIFEST)
        R.summarize(out_path, rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                    manifest=MANIFEST)

# (b) head-to-head against whichever other arms have finished. Scored on the
# presentations both arms searched — with sessions finishing at different times,
# an intersection is the only denominator both have earned.
#
# Read the COST columns, not just the solve count. Both shipped arms already
# reach 640/640 and 638/640 at this budget, so a refined arm has almost no room
# to add solves and will look inert on that column by construction; what it can
# move is nodes, and rank 1. The pairs worth naming:
#
#   abel vs abel_len_rd_s   THE HEADLINE — the shipped arm against the rule, and
#                           the only pair whose left half is already frozen at
#                           this budget. Read cost and rank 1: abel's rank 1
#                           takes 640/640 here, so the solve column cannot move.
#   len  vs len_rd_s        the same on the length arm, where solves CAN move:
#                           len's rank 1 fails on 7 rows and its top 3 on 2.
#   abel_len_rd vs abel_len_rd_s   what S alone is worth — needs the control arm
#   len_rd vs len_rd_s             run too, which this session does not do.
#
# At budget 1,000 over all 640 the same two pairs gave: abel 584 -> 590 at rank 1
# with 169k -> 162k deployed nodes but 249k -> 334k as run, and len 587 -> 588 at
# top 3. Expect cost to be the column that moves here, not solves.
def _arm(rule):
    hits = [p for p in glob.glob(os.path.join(LOCAL_OUT, f"{rule}top{K}_{BUDGET}_*.jsonl"))
            if not R._CHUNK_MARK.search(os.path.basename(p))]
    return max(hits, key=lambda p: sum(1 for _ in open(p))) if hits else None

# a pair whose files are not both present prints a skip line and costs nothing,
# so listing the ones this session cannot answer is deliberate, not an oversight
PAIRS = [("abel", "abel_len_rd_s"), ("len", "len_rd_s"),
         ("abel_len_rd", "abel_len_rd_s"), ("len_rd", "len_rd_s"),
         ("abel", "abel_rd"), ("len", "len_rd")]
for a_rule, b_rule in PAIRS:
    a, b = _arm(a_rule), _arm(b_rule)
    if a and b:
        print(f"\n===== {a_rule}  vs  {b_rule} =====")
        R.compare_rules(a, b, k=K, label_a=a_rule, label_b=b_rule)
    else:
        absent = [r for r, p in ((a_rule, a), (b_rule, b)) if not p]
        print(f"skip {a_rule} vs {b_rule}: no results file yet for {absent}")
